# LLM Zoomcamp — Module 2 Homework: Vector Search


In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
from embedder import Embedder

embedder = Embedder()

## Q1. Embedding a query


In [ ]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Vector shape: {v.shape}")
print(f"Q1 answer — v[0] = {v[0]:.4f}")

## Load documents


In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

## Q2. Cosine similarity


In [ ]:
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target = next(d for d in documents if d["filename"] == target_file)

v_doc = embedder.encode(target["content"])

# Both vectors are normalized, so dot product == cosine similarity
similarity = float(v.dot(v_doc))
print(f"Q2 answer — cosine similarity = {similarity:.4f}")

## Q3. Chunking and search by hand


In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

In [ ]:
from tqdm import tqdm

batch_size = 32
all_embeddings = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [c["content"] for c in chunks[i:i + batch_size]]
    all_embeddings.append(embedder.encode_batch(batch))

X = np.vstack(all_embeddings)
print(f"Embedding matrix shape: {X.shape}")

In [ ]:
scores = X.dot(v)
best_idx = int(np.argmax(scores))

print(f"Q3 answer — best chunk filename: {chunks[best_idx]['filename']}")
print(f"  score: {scores[best_idx]:.4f}")

## Q4. Vector search with minsearch


In [ ]:
from minsearch import VectorSearch

# fit(vectors_2d_array, payload_list) — vectors and docs are separate
vs = VectorSearch(keyword_fields=["filename"])
vs.fit(X, chunks)
print("VectorSearch index built")

In [ ]:
q4_query = "What metric do we use to evaluate a search engine?"
v_q4 = embedder.encode(q4_query)

# search(query_vector_numpy_array)
results_q4 = vs.search(v_q4, num_results=5)

print("Q4 answer — top 5 results:")
for i, r in enumerate(results_q4):
    print(f"  {i+1}. {r['filename']}")
print(f"\nQ4 answer — first result: {results_q4[0]['filename']}")

## Q5. Text search vs vector search


In [ ]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)
print("Text index built")

In [ ]:
q5_query = "How do I store vectors in PostgreSQL?"
v_q5 = embedder.encode(q5_query)

vector_results_q5 = vs.search(v_q5, num_results=5)
text_results_q5   = text_index.search(q5_query, num_results=5)

vector_files = {r["filename"] for r in vector_results_q5}
text_files   = {r["filename"] for r in text_results_q5}

print("Vector search top 5:")
for r in vector_results_q5:
    print(f"  {r['filename']}")

print("\nText search top 5:")
for r in text_results_q5:
    print(f"  {r['filename']}")

only_in_vector = vector_files - text_files
print(f"\nQ5 answer — in vector but not text: {only_in_vector}")

## Q6. Hybrid search with RRF


In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
q6_query = "How do I give the model access to tools?"
v_q6 = embedder.encode(q6_query)

vector_results_q6 = vs.search(v_q6, num_results=5)
text_results_q6   = text_index.search(q6_query, num_results=5)

print("Vector search top 5:")
for i, r in enumerate(vector_results_q6):
    print(f"  {i}. {r['filename']}")

print("\nText search top 5:")
for i, r in enumerate(text_results_q6):
    print(f"  {i}. {r['filename']}")

rrf_results = rrf([vector_results_q6, text_results_q6])

print("\nRRF fused top 5:")
for i, r in enumerate(rrf_results):
    print(f"  {i}. {r['filename']}")

print(f"\nQ6 answer — first result after RRF: {rrf_results[0]['filename']}")